# ViVQA-S1 Heavy Compute Offload Notebook (Google Colab GPU)

1. **Upload / Mount**: Upload `/home/khaitran/data/colab_boost_upload.zip` (`7.14 MB`) to `/content/` (or mount Google Drive containing `ezy_results`).
2. **Task 1 (`ref_heads`)**: Runs GPU-vectorized 3-seed Decision Head training (`head_s0..s2`, both `linear` and `res_mlp`, using 5-fold Out-of-Fold $W_0$-anchored adaptation on `calib` when `train` shards are absent, or full `train` shards when present) + Letter-Debiased `zeroshot` (`--debias-letters`) on **`ref-qwen3vl8b` (8.77B)**, **`ref-intern4b` (4.54B)**, and **`ref-intern2b` (2.21B)**.
3. **Task 2 (`rescore_legacy`)**: Re-scores the 5 legacy 1B–3B models (`onejev`, `jpt`, `intern`, `qwen`, `vintern`) on the canonical `43,903`-item (`N_test=4,409`) splits.
4. **Task 3 (`viso_v5`)**: Trains `ViSO Cross-Attn v5` with per-type best-checkpoint splicing (`TRAINING.SAVE_BEST_PER_TYPE=True`).
5. **Task 4 (`bundle`)**: Packages **only** the lightweight (`~180 KB`) `logits_{calib,dev,test}.npz`, `views.json`, and `calibration/` files into `colab_boost_results.zip` (`<15 MB`) and triggers browser download.

In [ ]:
# 1. Mount Google Drive & Verify GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
# 2. Unpack colab_boost_upload.zip (7.14 MB) & Configure Paths
import os, zipfile

UPLOAD_ZIP = '/content/colab_boost_upload.zip' if os.path.isfile('/content/colab_boost_upload.zip') else '/content/drive/MyDrive/colab_boost_upload.zip'
REPO_DIR = '/content/IE307-Final'
if os.path.isfile(UPLOAD_ZIP) and not os.path.isdir(REPO_DIR):
    with zipfile.ZipFile(UPLOAD_ZIP, 'r') as zf:
        zf.extractall(REPO_DIR)
    print('Extracted', UPLOAD_ZIP, 'to', REPO_DIR)
elif not os.path.isdir(REPO_DIR):
    REPO_DIR = '/content/drive/MyDrive/IE307-Final'

EZY_DIR  = '/content/drive/MyDrive/ezy_results'   # Update if your shards are stored elsewhere on Drive/Kaggle
S1_DATA  = f'{REPO_DIR}/vivqa_s1_v2/data'
VISO_OUT = '/content/drive/MyDrive/viso_v5_out'
OUT_ZIP  = '/content/colab_boost_results.zip'

os.chdir(f'{REPO_DIR}/ViVQA_S1')
print('Working directory:', os.getcwd())

In [ ]:
# 3. Task 1: Train 3-Seed Decision Heads (Linear + Residual MLP) on 4B-8B Reference VLMs
!python kaggle/colab_heavy_boost.py \
  --task ref_heads \
  --ezy-dir "$EZY_DIR" \
  --s1-data "$S1_DATA" \
  --head-arch linear \
  --seeds 3

!python kaggle/colab_heavy_boost.py \
  --task ref_heads \
  --ezy-dir "$EZY_DIR" \
  --s1-data "$S1_DATA" \
  --head-arch res_mlp \
  --seeds 3

In [ ]:
# 4. Task 2: Re-score Legacy 1B-3B Models on Canonical N=4,409 Splits with Letter Debiasing
!python kaggle/colab_heavy_boost.py \
  --task rescore_legacy \
  --ezy-dir "$EZY_DIR" \
  --s1-data "$S1_DATA" \
  --seeds 3

In [ ]:
# 5. Task 3: Train ViSO Cross-Attn v5 with Per-Type Best Checkpoint Splicing (SAVE_BEST_PER_TYPE=True)
!python kaggle/colab_heavy_boost.py \
  --task viso_v5 \
  --s1-data "$S1_DATA" \
  --viso-out "$VISO_OUT"

In [ ]:
# 6. Task 4: Bundle Only Lightweight (~180 KB) logits_{calib,dev,test}.npz & views.json (<15 MB total)
!python kaggle/colab_heavy_boost.py \
  --task bundle \
  --ezy-dir "$EZY_DIR" \
  --viso-out "$VISO_OUT" \
  --out-zip "$OUT_ZIP"

from google.colab import files
files.download(OUT_ZIP)